Import library

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

 Load data Excel

In [2]:
file_path = "Responses fix 2222.xlsx"
sheet_name = "Form Responses 1"

df_raw = pd.read_excel(file_path, sheet_name=sheet_name)

# Rapikan nama kolom
df_raw.columns = df_raw.columns.astype(str).str.strip()

print("Shape awal:", df_raw.shape)
print("Daftar kolom:")
for col in df_raw.columns:
    print("-", col)

Shape awal: (57, 24)
Daftar kolom:
- No
- Usia Anda saat ini
- Jenis kelamin
- Status Anda saat ini
- Apakah Anda pernah menggunakan SPayLater
- Apakah Anda pernah menggunakan SPayLater untuk membeli produk fashion (pakaian, sepatu, tas)?
- Apakah Anda pernah menggunakan SPayLater untuk membeli makanan atau minuman yang sedang viral di media sosial?
- Apakah Anda pernah menggunakan SPayLater untuk membeli produk skincare atau kosmetik?
- Apakah Anda pernah menggunakan SPayLater untuk hiburan seperti travelling, konser, atau rekreasi?
- Apakah Anda pernah menggunakan SPayLater untuk membayar internet atau tagihan bulanan?
- Saya merasa khawatir jika tertinggal tren yang sedang populer di media sosial.
- Saya tertarik membeli produk yang sedang viral di media sosial.
- Saya cenderung ingin membeli suatu barang setelah melihat orang lain memilikinya.
- Saya merasa takut kehilang kesempatan ketika ada promo atau diskon online.
- Saya menyisihkan sebagian pendapatan saya untuk ditabung
- Sa

Data Cleaning

In [3]:
drop_cols = [col for col in df_raw.columns if col.startswith("Unnamed")] + ["Email Address"]
df_raw = df_raw.drop(columns=drop_cols, errors="ignore")

df_raw = df_raw.loc[df_raw["No"].notna()].copy()

print("Shape setelah cleaning awal:", df_raw.shape)
df_raw.head()

Shape setelah cleaning awal: (51, 19)


,No,Usia Anda saat ini,Jenis kelamin,Status Anda saat ini,Apakah Anda pernah menggunakan SPayLater,"Apakah Anda pernah menggunakan SPayLater untuk membeli produk fashion (pakaian, sepatu, tas)?",Apakah Anda pernah menggunakan SPayLater untuk membeli makanan atau minuman yang sedang viral di media sosial?,Apakah Anda pernah menggunakan SPayLater untuk membeli produk skincare atau kosmetik?,"Apakah Anda pernah menggunakan SPayLater untuk hiburan seperti travelling, konser, atau rekreasi?",Apakah Anda pernah menggunakan SPayLater untuk membayar internet atau tagihan bulanan?,Saya merasa khawatir jika tertinggal tren yang sedang populer di media sosial.,Saya tertarik membeli produk yang sedang viral di media sosial.,Saya cenderung ingin membeli suatu barang setelah melihat orang lain memilikinya.,Saya merasa takut kehilang kesempatan ketika ada promo atau diskon online.,Saya menyisihkan sebagian pendapatan saya untuk ditabung,Saya membuat perencanaan keuangan sebelum melakukan pembelian,Saya mempertimbangkan kondisi keuangan sebelum menggunakan layanan paylater,Saya berusaha menabung secara rutin setiap bulan.,Saya menghindari pembelian barang yang tidak terlalu dibutuhkan.
0,1,18-29 tahun,Perempuan,Pelajar / Mahasiswa,Ya,Ya,Tidak,Tidak,Tidak,Ya,2.0,3.0,3.0,4.0,4.0,3.0,3.0,3.0,4.0
1,2,18-29 tahun,Perempuan,Pelajar / Mahasiswa,Ya,Ya,Ya,Ya,Tidak,Tidak,4.0,5.0,4.0,4.0,4.0,4.0,3.0,5.0,4.0
2,3,18-29 tahun,Perempuan,Pelajar / Mahasiswa,Ya,Ya,Ya,Ya,Ya,Ya,5.0,5.0,5.0,5.0,2.0,1.0,1.0,2.0,1.0
3,4,18-29 tahun,Laki-laki,Pelajar / Mahasiswa,Ya,Tidak,Tidak,Tidak,Tidak,Ya,1.0,1.0,1.0,1.0,4.0,3.0,4.0,4.0,4.0
4,5,18-29 tahun,Perempuan,Pelajar / Mahasiswa,Ya,Ya,Tidak,Tidak,Tidak,Tidak,3.0,4.0,1.0,2.0,4.0,3.0,5.0,3.0,5.0


In [7]:
# Ubah kolom Likert ke numerik
for col in fomo_cols + financial_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Mapping jawaban Ya/Tidak
binary_map = {"Ya": 1, "Tidak": 0}

for col in [col_fashion, col_viral_food, col_skincare, col_hiburan, col_tagihan]:
    df[col] = df[col].map(binary_map)

# Hapus baris yang masih kosong pada kolom penting
df = df.dropna().copy()

print("Shape setelah dropna:", df.shape)
df.head()

Shape setelah dropna: (50, 16)


,Jenis kelamin,Status Anda saat ini,"Apakah Anda pernah menggunakan SPayLater untuk membeli produk fashion (pakaian, sepatu, tas)?",Apakah Anda pernah menggunakan SPayLater untuk membeli makanan atau minuman yang sedang viral di media sosial?,Apakah Anda pernah menggunakan SPayLater untuk membeli produk skincare atau kosmetik?,"Apakah Anda pernah menggunakan SPayLater untuk hiburan seperti travelling, konser, atau rekreasi?",Apakah Anda pernah menggunakan SPayLater untuk membayar internet atau tagihan bulanan?,Saya merasa khawatir jika tertinggal tren yang sedang populer di media sosial.,Saya tertarik membeli produk yang sedang viral di media sosial.,Saya cenderung ingin membeli suatu barang setelah melihat orang lain memilikinya.,Saya merasa takut kehilang kesempatan ketika ada promo atau diskon online.,Saya menyisihkan sebagian pendapatan saya untuk ditabung,Saya membuat perencanaan keuangan sebelum melakukan pembelian,Saya mempertimbangkan kondisi keuangan sebelum menggunakan layanan paylater,Saya berusaha menabung secara rutin setiap bulan.,Saya menghindari pembelian barang yang tidak terlalu dibutuhkan.
0,Perempuan,Pelajar / Mahasiswa,1.0,0.0,0.0,0.0,1.0,2.0,3.0,3.0,4.0,4.0,3.0,3.0,3.0,4.0
1,Perempuan,Pelajar / Mahasiswa,1.0,1.0,1.0,0.0,0.0,4.0,5.0,4.0,4.0,4.0,4.0,3.0,5.0,4.0
2,Perempuan,Pelajar / Mahasiswa,1.0,1.0,1.0,1.0,1.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,2.0,1.0
3,Laki-laki,Pelajar / Mahasiswa,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,4.0,3.0,4.0,4.0,4.0
4,Perempuan,Pelajar / Mahasiswa,1.0,0.0,0.0,0.0,0.0,3.0,4.0,1.0,2.0,4.0,3.0,5.0,3.0,5.0


In [8]:
df["fomo_score"] = df[fomo_cols].mean(axis=1)
df["financial_score"] = df[financial_cols].mean(axis=1)

# Target utama: penggunaan SPayLater untuk membeli makanan/minuman viral
df["target_viral_food"] = df[col_viral_food].astype(int)

# Dataset final untuk model
model_df = df[["fomo_score", "financial_score", "target_viral_food"]].copy()

print(model_df.head())
print("\nDistribusi target:")
print(model_df["target_viral_food"].value_counts())

   fomo_score  financial_score  target_viral_food
0        3.00              3.4                  0
1        4.25              4.0                  1
2        5.00              1.4                  1
3        1.00              3.8                  0
4        2.50              4.0                  0

Distribusi target:
target_viral_food
1    28
0    22
Name: count, dtype: int64


In [9]:
print("Jumlah responden:", len(model_df))
print("Rata-rata FOMO:", round(model_df["fomo_score"].mean(), 2))
print("Rata-rata Literasi Keuangan:", round(model_df["financial_score"].mean(), 2))
print("Persentase target Ya:", round(model_df["target_viral_food"].mean() * 100, 2), "%")

Jumlah responden: 50
Rata-rata FOMO: 3.13
Rata-rata Literasi Keuangan: 4.09
Persentase target Ya: 56.0 %


train dan test split data

In [10]:
X = model_df[["fomo_score", "financial_score"]]
y = model_df["target_viral_food"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (40, 2)
Test size: (10, 2)


Logistic Regression

In [11]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])

pipeline.fit(X_train, y_train)

print("Model berhasil dilatih")

Model berhasil dilatih


Evaluation

In [12]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

print("Accuracy :", round(acc, 4))
print("ROC AUC  :", round(auc, 4))
print("Confusion Matrix:")
print(cm)

Accuracy : 0.8
ROC AUC  : 0.6667
Confusion Matrix:
[[2 2]
 [0 6]]


In [13]:
coef_fomo = pipeline.named_steps["model"].coef_[0][0]
coef_financial = pipeline.named_steps["model"].coef_[0][1]
intercept = pipeline.named_steps["model"].intercept_[0]

print("Koefisien fomo_score      :", round(coef_fomo, 4))
print("Koefisien financial_score :", round(coef_financial, 4))
print("Intercept                 :", round(intercept, 4))

Koefisien fomo_score      : 0.5609
Koefisien financial_score : -0.2749
Intercept                 : 0.2093


In [14]:
dashboard_df = df[[
    col_gender,
    col_status,
    "fomo_score",
    "financial_score",
    col_fashion,
    col_viral_food,
    col_skincare,
    col_hiburan,
    col_tagihan,
    "target_viral_food"
]].copy()

dashboard_df = dashboard_df.rename(columns={
    col_gender: "gender",
    col_status: "status",
    col_fashion: "fashion",
    col_viral_food: "viral_food",
    col_skincare: "skincare",
    col_hiburan: "hiburan",
    col_tagihan: "tagihan"
})

dashboard_df.head()

,gender,status,fomo_score,financial_score,fashion,viral_food,skincare,hiburan,tagihan,target_viral_food
0,Perempuan,Pelajar / Mahasiswa,3.00,3.4,1.0,0.0,0.0,0.0,1.0,0
1,Perempuan,Pelajar / Mahasiswa,4.25,4.0,1.0,1.0,1.0,0.0,0.0,1
2,Perempuan,Pelajar / Mahasiswa,5.00,1.4,1.0,1.0,1.0,1.0,1.0,1
3,Laki-laki,Pelajar / Mahasiswa,1.00,3.8,0.0,0.0,0.0,0.0,1.0,0
4,Perempuan,Pelajar / Mahasiswa,2.50,4.0,1.0,0.0,0.0,0.0,0.0,0


Save Model

In [15]:
os.makedirs("artifacts", exist_ok=True)

# Simpan model
joblib.dump(pipeline, "artifacts/spaylater_logreg.pkl")

# Simpan data dashboard
dashboard_df.to_csv("artifacts/dashboard_spaylater.csv", index=False)

# Simpan metrik model
metrics = {
    "accuracy": float(acc),
    "roc_auc": float(auc),
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "coefficients": {
        "fomo_score": float(coef_fomo),
        "financial_score": float(coef_financial)
    },
    "intercept": float(intercept),
    "n_rows": int(len(dashboard_df))
}

with open("artifacts/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Semua artifact berhasil disimpan ke folder artifacts/")

Semua artifact berhasil disimpan ke folder artifacts/
